In [1]:
import warnings
import pandas as pd
from analysis_utils_msr import assess_grouped_spearman
from tqdm.notebook import tqdm
import numpy as np

# notebook display options
warnings.filterwarnings('ignore')

In [2]:
def add_additive_predictions(df_preds, df_ref):
    """
    Calculates additive predictions for multimutants in df_ref by summing 
    single-mutant predictions from df_preds.

    Args:
        df_preds (pd.DataFrame): Dataframe containing single mutant predictions.
                                 Must have columns: ['code', 'mut_type', 'ddG_pred']
        df_ref (pd.DataFrame):   Reference dataframe containing multimutants 
                                 (colon-separated).
                                 Must have columns: ['code', 'mut_type']

    Returns:
        pd.DataFrame: A copy of df_ref with a new 'ddG_pred_additive' column.
    """
    # 1. Prepare the reference dataframe
    # We create a temporary ID to ensure we can group back to the exact original rows later
    df_out = df_ref.copy()
    df_out['_temp_id'] = df_out.index

    # 2. Explode the multimutants in the reference df
    # Select only necessary columns to keep the operation lightweight
    exploded = df_out[['_temp_id', 'code', 'mut_type']].copy()
    
    # Split "A1G:T2C" -> ["A1G", "T2C"]
    exploded['single_mut'] = exploded['mut_type'].str.split(':')
    
    # Explode into separate rows: one row per single mutation
    exploded = exploded.explode('single_mut')

    # 3. Merge with the predictions
    # We perform a left join to attach the 'ddG_pred' from df_preds to our exploded list
    # matching on 'code' (pdb_id) and the specific mutation
    merged = exploded.merge(
        df_preds[['code', 'mut_type', 'ddG_pred']],
        left_on=['code', 'single_mut'],
        right_on=['code', 'mut_type'],
        how='left'
    )

    # 4. Sum the predictions
    # We group by the temporary ID (original row) and sum the ddG values.
    # min_count=1 ensures that if a mutation is missing (NaN), the result is NaN 
    # (or partial sum) rather than blindly returning 0.
    additive_sums = merged.groupby('_temp_id')['ddG_pred'].sum(min_count=1)

    # 5. Assign result back to the reference dataframe
    df_out['ddG_pred_additive'] = additive_sums
    
    # Cleanup
    del df_out['_temp_id']
    
    return df_out

In [3]:
# Get all raw predictions

dataset = 'feb19_tsuboyama-test'

df_ts_1_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new1_epoch=99_val_ddG_spearman=0.71.ckpt_singles.csv', index_col=0)
df_ts_1_single['ThermoMPNN(-D)_1'] = df_ts_1_single['ddG_pred']
df_ts_2_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new2_epoch=99_val_ddG_spearman=0.71.ckpt_singles.csv', index_col=0)
df_ts_2_single['ThermoMPNN(-D)_2'] = df_ts_2_single['ddG_pred']
df_ts_3_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new3_epoch=98_val_ddG_spearman=0.7.ckpt_singles.csv', index_col=0)
df_ts_3_single['ThermoMPNN(-D)_3'] = df_ts_3_single['ddG_pred']

df_ts_1_epistatic = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/epistatic_feb19_tsuboyama_new1_epoch=44_val_ddG_spearman=0.71.ckpt.csv', index_col=0)
df_ts_1_epistatic['ThermoMPNN(-D)_1'] = df_ts_1_epistatic['ddG_pred']
df_ts_2_epistatic = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/epistatic_feb19_tsuboyama_new2_epoch=82_val_ddG_spearman=0.7.ckpt.csv', index_col=0)
df_ts_2_epistatic['ThermoMPNN(-D)_2'] = df_ts_2_epistatic['ddG_pred']
df_ts_3_epistatic = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/epistatic_feb19_tsuboyama_new3_epoch=71_val_ddG_spearman=0.7.ckpt.csv', index_col=0)
df_ts_3_epistatic['ThermoMPNN(-D)_3'] = df_ts_3_epistatic['ddG_pred']

df_ts_1_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new1_epoch=99_val_ddG_spearman=0.71.ckpt_singles.csv', index_col=0)
df_ts_1_single['ThermoMPNN_1'] = df_ts_1_single['ddG_pred']
df_ts_2_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new2_epoch=99_val_ddG_spearman=0.71.ckpt_singles.csv', index_col=0)
df_ts_2_single['ThermoMPNN_2'] = df_ts_2_single['ddG_pred']
df_ts_3_single = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/single_feb19_tsuboyama_new3_epoch=98_val_ddG_spearman=0.7.ckpt_singles.csv', index_col=0)
df_ts_3_single['ThermoMPNN_3'] = df_ts_3_single['ddG_pred']

df_ts_1_additive = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/additive_single_feb19_tsuboyama_new1_epoch=99_val_ddG_spearman=0.71.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')
df_ts_1_additive['ThermoMPNN_1'] = df_ts_1_additive['ddG_pred_additive']
df_ts_2_additive = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/additive_single_feb19_tsuboyama_new2_epoch=99_val_ddG_spearman=0.71.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')
df_ts_2_additive['ThermoMPNN_2'] = df_ts_2_additive['ddG_pred_additive']
df_ts_3_additive = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/thermompnnd/additive_single_feb19_tsuboyama_new3_epoch=98_val_ddG_spearman=0.7.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')
df_ts_3_additive['ThermoMPNN_3'] = df_ts_3_additive['ddG_pred_additive']

df_ts_1_epi = pd.concat([df_ts_1_single.rename({'ThermoMPNN_1': 'ThermoMPNN(-D)_1'}, axis=1), df_ts_1_epistatic])
df_ts_2_epi = pd.concat([df_ts_2_single.rename({'ThermoMPNN_2': 'ThermoMPNN(-D)_2'}, axis=1), df_ts_2_epistatic])
df_ts_3_epi = pd.concat([df_ts_3_single.rename({'ThermoMPNN_3': 'ThermoMPNN(-D)_3'}, axis=1), df_ts_3_epistatic])

df_ts_1_add = pd.concat([df_ts_1_single, df_ts_1_additive])
df_ts_2_add = pd.concat([df_ts_2_single, df_ts_2_additive])
df_ts_3_add = pd.concat([df_ts_3_single, df_ts_3_additive])

df_ts_1 = df_ts_1_add.join(df_ts_1_epi[['ThermoMPNN(-D)_1']])
df_ts_2 = df_ts_2_add.join(df_ts_2_epi[['ThermoMPNN(-D)_2']])
df_ts_3 = df_ts_3_add.join(df_ts_3_epi[['ThermoMPNN(-D)_3']])

df_thermo = df_ts_1[['code', 'mut_type', 'ThermoMPNN_1', 'ThermoMPNN(-D)_1', 'ddG_true']].join(df_ts_2[['ThermoMPNN_2', 'ThermoMPNN(-D)_2']], how='inner').join(df_ts_3[['ThermoMPNN_3', 'ThermoMPNN(-D)_3']], how='inner')
df_thermo = df_thermo.rename({'ddG_true': 'ddG_ML_thermo'}, axis=1)

df_me_1 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_new1_feb19_tsuboyama_checkpoint-99.csv', index_col=0)
df_me_1['MutateEverything_1'] = df_me_1['ddG_pred']
df_me_1['ddG_ML_me'] = df_me_1['ddG_true']
df_me_2 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_new2_feb19_tsuboyama_checkpoint-99.csv', index_col=0)
df_me_2['MutateEverything_2'] = df_me_2['ddG_pred']
df_me_3 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_new3_feb19_tsuboyama_checkpoint-99.csv', index_col=0)
df_me_3['MutateEverything_3'] = df_me_3['ddG_pred']

df_me = df_me_1[['code', 'mut_type', 'MutateEverything_1', 'ddG_ML_me']].join(df_me_2[['MutateEverything_2']], how='inner').join(df_me_3[['MutateEverything_3']], how='inner')
if dataset in 'domainome':
        df_me.index = df_me['code'].str.rstrip('A') + '_' + df_me['mut_type']
assert len(df_me) == len(df_me_1)

df_me_additive_1 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_additive_new1_feb19_tsuboyama_checkpoint-99_additive.csv', index_col=0)
df_me_additive_1 = add_additive_predictions(df_me_additive_1, df_me)
df_me_additive_1 = df_me_additive_1[['code', 'mut_type', 'ddG_pred_additive']].rename({'ddG_pred_additive': 'MutateEverything_additive_1'}, axis=1)
df_me_additive_2 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_additive_new2_feb19_tsuboyama_checkpoint-99_additive.csv', index_col=0)
df_me_additive_2 = add_additive_predictions(df_me_additive_2, df_me)[['code', 'mut_type', 'ddG_pred_additive']].rename({'ddG_pred_additive': 'MutateEverything_additive_2'}, axis=1)
df_me_additive_3 = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/mutate_everything/dms_additive_new3_feb19_tsuboyama_checkpoint-99_additive.csv', index_col=0)
df_me_additive_3 = add_additive_predictions(df_me_additive_3, df_me)[['code', 'mut_type', 'ddG_pred_additive']].rename({'ddG_pred_additive': 'MutateEverything_additive_3'}, axis=1)

df_me_additive = df_me_additive_1[['MutateEverything_additive_1']].join(df_me_additive_2[['MutateEverything_additive_2']], how='inner').join(df_me_additive_3[['MutateEverything_additive_3']], how='inner')
assert len(df_me_additive) == len(df_me), 'e'
df_me = df_me.join(df_me_additive)

df_zero_sm = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/zero_shot/esm3-small-2024-08_masked.csv', index_col=0).rename({'esm3_score': 'ESM3-small', 'additive_score': 'ESM3-small_additive'}, axis=1)
df_zero_med = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/zero_shot/esm3-medium-2024-08_masked.csv', index_col=0).rename({'esm3_score': 'ESM3-medium', 'additive_score': 'ESM3-medium_additive'}, axis=1)
df_zero_lg = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/zero_shot/esm3-large-2024-03_masked.csv', index_col=0).rename({'esm3_score': 'ESM3-large', 'additive_score': 'ESM3-large_additive'}, axis=1)

df_mpnn = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}/proteinmpnn/proteinmpnn_020_chain.csv', index_col=0).rename({'mpnn_score': 'ProteinMPNN', 'mpnn_score_additive': 'ProteinMPNN_additive'}, axis=1)

#df_ros = pd.read_csv('/home/sareeves/PSLMs/analysis_notebooks/predictions_good_may9/rosetta_all_preds_april17.csv')
#df_ros = df_ros.rename({'cartesian_ddg_dir': 'Rosetta Cartesian DDG'}, axis=1)
#df_ros = df_ros[['uid', 'code', 'mut_type', 'Rosetta Cartesian DDG', 'ddG_ML']]
#df_ros = df_ros.set_index('uid')
#df_ros = df_ros.loc[list(set(df_me.index).intersection(set(df_ros.index)))]

df_ros = pd.read_csv(f'~/software/esm-msr/analysis_notebooks/predictions/tsuboyama_all/cartesian_ddg_feb22.csv', index_col=0)
df_ros['Rosetta Cartesian DDG_1'] = df_ros['cartesian_ddg_1']
#df_ros['Rosetta Cartesian DDG_additive_1'] = df_ros['cartesian_ddg_additive']
df_ros['Rosetta Cartesian DDG_2'] = df_ros['cartesian_ddg_2']
#df_ros['Rosetta Cartesian DDG_additive_2'] = df_ros['cartesian_ddg_2_additive']
df_ros['Rosetta Cartesian DDG_3'] = df_ros['cartesian_ddg_3']
#df_ros['Rosetta Cartesian DDG_additive_3'] = df_ros['cartesian_ddg_3_additive']
df_ros.index.name = 'uid'
df_ros = df_ros.reset_index()
df_ros['code'] = df_ros['uid'].apply(lambda x: x.split('_')[0])
df_ros['mut_type'] = df_ros['uid'].apply(lambda x: x.split('_')[-1])
df_ros = df_ros.set_index('uid')

df = \
df_zero_sm[[c for c in df_zero_sm.columns if 'ESM3' in c]].join(
df_thermo[[c for c in df_thermo.columns if 'ThermoMPNN' in c] + ['ddG_ML_thermo']], how='left').join(
df_me[[c for c in df_me.columns if 'MutateEverything' in c] + ['ddG_ML_me']], how='left').join(
#df_esm[[c for c in df_esm.columns if 'ESM-MSR' in c] + ['ddG_ML_esm', 'code', 'mut_type']], how='left').join(
df_zero_med[[c for c in df_zero_med.columns if 'ESM3' in c]], how='left').join(
df_zero_lg[[c for c in df_zero_lg.columns if 'ESM3' in c]], how='left').join(
df_ros[[c for c in df_ros.columns if 'Rosetta' in c]], how='left').join(
df_mpnn[[c for c in df_mpnn.columns if 'ProteinMPNN' in c]], how='left'
)

truth = 'ddG_ML'

df_esm_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/{dataset}/msr_chain/1/epoch=08-val_rho_avg=0.758.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_1.loc[~df_esm_1['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_1.loc[~df_esm_1['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_1['ESM-MSR_1'] = df_esm_1['ddg_pred']
df_esm_1['ESM-MSR_additive_1'] = df_esm_1['ddg_pred_additive']
#print(df_esm_1[['ESM-MSR_1', 'ESM-MSR_additive_1']])
df_esm_1['ddG_ML_esm'] = df_esm_1[truth]

df_esm_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/{dataset}/msr_chain/2/epoch=07-val_rho_avg=0.755.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_2.loc[~df_esm_2['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_2.loc[~df_esm_2['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_2['ESM-MSR_2'] = df_esm_2['ddg_pred']
df_esm_2['ESM-MSR_additive_2'] = df_esm_2['ddg_pred_additive']

df_esm_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/{dataset}/msr_chain/3/epoch=08-val_rho_avg=0.757.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_3.loc[~df_esm_3['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_3.loc[~df_esm_3['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_3['ESM-MSR_3'] = df_esm_3['ddg_pred']
df_esm_3['ESM-MSR_additive_3'] = df_esm_3['ddg_pred_additive']
df_esm = pd.concat([df_esm_1, df_esm_2[[c for c in df_esm_2.columns if 'MSR' in c]], df_esm_3[[c for c in df_esm_3.columns if 'MSR' in c]]], axis=1)

df_esm_zs = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/{dataset}/None_alpha0chain_rule_avg_masked.csv', index_col=0).sort_index()
df_esm_zs['ESM3-small-open'] = df_esm_zs['ddg_pred']
df_esm_zs['ESM3-small-open_additive'] = df_esm_zs['ddg_pred_additive']

df = df.join(df_esm[[c for c in df_esm.columns if 'MSR' in c]+['code', 'mut_type', 'ddG_ML_esm']], how='left')
df = df.join(df_esm_zs[['ESM3-small-open'] + (['ESM3-small-open_additive'])], how='left')

df = df.rename({'ddG_ML_esm': 'ddG'}, axis=1)

df['random_1'] = np.random.rand(len(df))
df['random_2'] = np.random.rand(len(df))
df['random_3'] = np.random.rand(len(df))

# took estimated measurement error from Tsuboyama paper
df['ddG_1'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))
df['ddG_2'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))
df['ddG_3'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))

df

,ESM3-small,ESM3-small_additive,ThermoMPNN_1,ThermoMPNN(-D)_1,ThermoMPNN_2,ThermoMPNN(-D)_2,ThermoMPNN_3,ThermoMPNN(-D)_3,ddG_ML_thermo,MutateEverything_1,...,mut_type,ddG,ESM3-small-open,ESM3-small-open_additive,random_1,random_2,random_3,ddG_1,ddG_2,ddG_3
uid,,,,,,,,,,,,,,,,,,,,,
1TUD_A24C,-4.390625,NaN,-0.925135,-0.925135,-0.886917,-0.886917,-0.952761,-0.952761,-0.270375,-0.814228,...,A24C,-0.270375,-5.298828,NaN,0.477498,0.330715,0.961274,-0.439552,-0.146417,-0.369068
1TUD_A24F,-13.187500,NaN,-1.486816,-1.486816,-1.206404,-1.206404,-1.260494,-1.260494,-3.959452,-1.489653,...,A24F,-3.959452,-13.578125,NaN,0.010373,0.957597,0.341098,-3.973389,-3.934498,-3.875255
1TUD_A24G,-7.257812,NaN,-1.522053,-1.522053,-1.454631,-1.454631,-1.334050,-1.334050,-3.049759,-2.172791,...,A24G,-3.049759,-8.005859,NaN,0.374236,0.961852,0.577989,-3.113691,-3.223696,-2.995495
1TUD_A24I,-9.976562,NaN,-1.099106,-1.099106,-1.049073,-1.049073,-1.186555,-1.186555,-3.423371,-0.735876,...,A24I,-3.423371,-11.265625,NaN,0.930971,0.616163,0.179653,-3.579620,-3.323567,-3.397278
1TUD_A24K,-13.375000,NaN,-2.208153,-2.208153,-2.062830,-2.062830,-2.229763,-2.229763,-4.974819,-3.481638,...,A24K,-4.974819,-13.382812,NaN,0.370050,0.169419,0.268257,-4.923751,-5.099123,-4.840334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2L2P_R39Y:T46R,-8.032227,-9.728516,-1.581925,-1.701419,-1.973589,-1.444623,-1.756877,-1.526226,-0.312255,-0.755020,...,R39Y:T46R,-0.312255,-8.910156,-11.133789,0.222260,0.109656,0.629795,-0.360383,-0.258475,-0.265105
2L2P_R39Y:T46S,-8.964111,-9.817871,-1.718593,-1.737093,-2.012937,-1.476977,-1.912432,-1.452884,-0.735413,-1.126285,...,R39Y:T46S,-0.735413,-8.946289,-9.534180,0.280490,0.257581,0.071268,-0.980012,-0.821443,-0.824777
2L2P_R39Y:T46V,-8.656250,-7.921875,-0.951985,-1.136660,-1.252671,-0.864598,-1.024171,-0.541343,-0.416700,-0.067048,...,R39Y:T46V,-0.416700,-9.767822,-9.289062,0.681327,0.535362,0.355551,-0.618798,-0.423112,-0.610518


In [4]:
df_esm_seqonly_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_seq_only/2/epoch=08-val_rho_avg=0.566.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_1.loc[~df_esm_seqonly_1['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_seqonly_1.loc[~df_esm_seqonly_1['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_seqonly_1['ESM-MSR_seqonly_1'] = df_esm_seqonly_1['ddg_pred']
df_esm_seqonly_1['ESM-MSR_seqonly_additive_1'] = df_esm_seqonly_1['ddg_pred_additive']
print(df_esm_seqonly_1[['ESM-MSR_seqonly_1', 'ESM-MSR_seqonly_additive_1']])
df_esm_seqonly_1['ddG_ML_esm'] = df_esm_seqonly_1[truth]

df_esm_seqonly_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_seq_only/2/epoch=08-val_rho_avg=0.566.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_2.loc[~df_esm_seqonly_2['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_seqonly_2.loc[~df_esm_seqonly_2['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_seqonly_2['ESM-MSR_seqonly_2'] = df_esm_seqonly_2['ddg_pred']
df_esm_seqonly_2['ESM-MSR_seqonly_additive_2'] = df_esm_seqonly_2['ddg_pred_additive']

df_esm_seqonly_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_seq_only/3/epoch=07-val_rho_avg=0.563.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_3.loc[~df_esm_seqonly_3['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_seqonly_3.loc[~df_esm_seqonly_3['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_seqonly_3['ESM-MSR_seqonly_3'] = df_esm_seqonly_3['ddg_pred']
df_esm_seqonly_3['ESM-MSR_seqonly_additive_3'] = df_esm_seqonly_3['ddg_pred_additive']
df_esm_seqonly = pd.concat([df_esm_seqonly_1, df_esm_seqonly_2[[c for c in df_esm_seqonly_2.columns if 'MSR' in c]], df_esm_seqonly_3[[c for c in df_esm_seqonly_3.columns if 'MSR' in c]]], axis=1)

df_esm_seqonly

df_esm_seqonly_domainome_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_seq_only/2/epoch=08-val_rho_avg=0.566.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_domainome_1['ESM-MSR_seqonly_1'] = df_esm_seqonly_domainome_1['ddg_pred']
df_esm_seqonly_domainome_1['id'] = df_esm_seqonly_domainome_1['code'] + '_' + df_esm_seqonly_domainome_1['mut_type']
df_esm_seqonly_domainome_1 = df_esm_seqonly_domainome_1.set_index('id')

df_esm_seqonly_domainome_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_seq_only/2/epoch=08-val_rho_avg=0.566.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_domainome_2['ESM-MSR_seqonly_2'] = df_esm_seqonly_domainome_2['ddg_pred']
#df_esm_seqonly_domainome_2['mut_type'] = df_esm_seqonly_domainome_2['mut_info']
df_esm_seqonly_domainome_2['id'] = df_esm_seqonly_domainome_2['code'] + '_' + df_esm_seqonly_domainome_2['mut_type']
df_esm_seqonly_domainome_2 = df_esm_seqonly_domainome_2.set_index('id')

df_esm_seqonly_domainome_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_seq_only/3/epoch=07-val_rho_avg=0.563.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_seqonly_domainome_3['ESM-MSR_seqonly_3'] = df_esm_seqonly_domainome_3['ddg_pred']
#df_esm_seqonly_domainome_3['mut_type'] = df_esm_seqonly_domainome_3['mut_info']
df_esm_seqonly_domainome_3['id'] = df_esm_seqonly_domainome_3['code'] + '_' + df_esm_seqonly_domainome_3['mut_type']
df_esm_seqonly_domainome_3 = df_esm_seqonly_domainome_3.set_index('id')

df_esm_seqonly_domainome = pd.concat([df_esm_seqonly_domainome_1, df_esm_seqonly_domainome_2[[c for c in df_esm_seqonly_domainome_2.columns if 'MSR' in c]], df_esm_seqonly_domainome_3[[c for c in df_esm_seqonly_domainome_3.columns if 'MSR' in c]]], axis=1)

df_esm_seqonly_domainome

           ESM-MSR_seqonly_1  ESM-MSR_seqonly_additive_1
uid                                                     
1TUD_A24C          -0.984433                   -0.984433
1TUD_A24F          -3.703111                   -3.703111
1TUD_A24G          -2.598573                   -2.598573
1TUD_A24I          -3.406814                   -3.406814
1TUD_A24K          -3.553149                   -3.553149
...                      ...                         ...
2L2P_Y9R           -1.903257                   -1.903257
2L2P_Y9S           -1.974762                   -1.974762
2L2P_Y9T           -3.075445                   -3.075445
2L2P_Y9V           -2.342109                   -2.342109
2L2P_Y9W           -0.626458                   -0.626458

[190559 rows x 2 columns]


,code,mut_seq,mut_type,uniprot_ID,pdb_file,ddG_ML,chain,pdb,chain_id,wt1,pos1,mut1,delta_logit1,ddg_pred,scheme,ESM-MSR_seqonly_1,ESM-MSR_seqonly_2,ESM-MSR_seqonly_3
id,,,,,,,,,,,,,,,,,,
A0A2R8Y422_PF00240_2_A27C,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,A27C,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.025738,A,A0A2R8Y422_2-71.pdb,A,A,27,C,-0.363525,-0.252836,masked,-0.252836,-0.252836,-0.273149
A0A2R8Y422_PF00240_2_A27D,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,A27D,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.111544,A,A0A2R8Y422_2-71.pdb,A,A,27,D,-3.359375,-1.180352,masked,-1.180352,-1.180352,-1.202612
A0A2R8Y422_PF00240_2_A27E,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,A27E,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.290325,A,A0A2R8Y422_2-71.pdb,A,A,27,E,-1.590332,-0.632656,masked,-0.632656,-0.632656,-0.707002
A0A2R8Y422_PF00240_2_A27F,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,A27F,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.002225,A,A0A2R8Y422_2-71.pdb,A,A,27,F,-0.419067,-0.270032,masked,-0.270032,-0.270032,-0.337081
A0A2R8Y422_PF00240_2_A27G,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,A27G,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.033329,A,A0A2R8Y422_2-71.pdb,A,A,27,G,-3.041016,-1.081788,masked,-1.081788,-1.081788,-1.100693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6N9_PF00595_207_V83R,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83R,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.040825,A,Q9Y6N9_207-295.pdb,A,V,83,R,-1.918945,-0.734395,masked,-0.734395,-0.734395,-0.749727
Q9Y6N9_PF00595_207_V83S,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83S,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.228931,A,Q9Y6N9_207-295.pdb,A,V,83,S,-4.690430,-1.592447,masked,-1.592447,-1.592447,-1.638096
Q9Y6N9_PF00595_207_V83T,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83T,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.353383,A,Q9Y6N9_207-295.pdb,A,V,83,T,-3.257446,-1.148795,masked,-1.148795,-1.148795,-1.185075


In [5]:
df_esm_maskseq_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_masked_sequence/1/epoch=10-val_rho_avg=0.706.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_1.loc[~df_esm_maskseq_1['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_maskseq_1.loc[~df_esm_maskseq_1['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_maskseq_1['ESM-MSR_maskseq_1'] = df_esm_maskseq_1['ddg_pred']
df_esm_maskseq_1['ESM-MSR_maskseq_additive_1'] = df_esm_maskseq_1['ddg_pred_additive']
print(df_esm_maskseq_1[['ESM-MSR_maskseq_1', 'ESM-MSR_maskseq_additive_1']])
df_esm_maskseq_1['ddG_ML_esm'] = df_esm_maskseq_1[truth]

df_esm_maskseq_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_masked_sequence/2/epoch=08-val_rho_avg=0.701.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_2.loc[~df_esm_maskseq_2['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_maskseq_2.loc[~df_esm_maskseq_2['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_maskseq_2['ESM-MSR_maskseq_2'] = df_esm_maskseq_2['ddg_pred']
df_esm_maskseq_2['ESM-MSR_maskseq_additive_2'] = df_esm_maskseq_2['ddg_pred_additive']

df_esm_maskseq_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/{dataset}/msr_chain_masked_sequence/3/epoch=06-val_rho_avg=0.704.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_3.loc[~df_esm_maskseq_3['mut_type'].str.contains(':'), 'ddg_pred_additive'] = df_esm_maskseq_3.loc[~df_esm_maskseq_3['mut_type'].str.contains(':'), 'ddg_pred']
df_esm_maskseq_3['ESM-MSR_maskseq_3'] = df_esm_maskseq_3['ddg_pred']
df_esm_maskseq_3['ESM-MSR_maskseq_additive_3'] = df_esm_maskseq_3['ddg_pred_additive']
df_esm_maskseq = pd.concat([df_esm_maskseq_1, df_esm_maskseq_2[[c for c in df_esm_maskseq_2.columns if 'MSR' in c]], df_esm_maskseq_3[[c for c in df_esm_maskseq_3.columns if 'MSR' in c]]], axis=1)

df_esm_maskseq

df_esm_maskseq_domainome_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_masked_sequence/1/epoch=10-val_rho_avg=0.706.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_domainome_1['ESM-MSR_maskseq_1'] = df_esm_maskseq_domainome_1['ddg_pred']
df_esm_maskseq_domainome_1['id'] = df_esm_maskseq_domainome_1['code'] + '_' + df_esm_maskseq_domainome_1['mut_type']
df_esm_maskseq_domainome_1 = df_esm_maskseq_domainome_1.set_index('id')

df_esm_maskseq_domainome_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_masked_sequence/2/epoch=08-val_rho_avg=0.701.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_domainome_2['ESM-MSR_maskseq_2'] = df_esm_maskseq_domainome_2['ddg_pred']
#df_esm_maskseq_domainome_2['mut_type'] = df_esm_maskseq_domainome_2['mut_info']
df_esm_maskseq_domainome_2['id'] = df_esm_maskseq_domainome_2['code'] + '_' + df_esm_maskseq_domainome_2['mut_type']
df_esm_maskseq_domainome_2 = df_esm_maskseq_domainome_2.set_index('id')

df_esm_maskseq_domainome_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/inference_scripts/predictions/domainome/msr_chain_masked_sequence/3/epoch=06-val_rho_avg=0.704.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_esm_maskseq_domainome_3['ESM-MSR_maskseq_3'] = df_esm_maskseq_domainome_3['ddg_pred']
#df_esm_maskseq_domainome_3['mut_type'] = df_esm_maskseq_domainome_3['mut_info']
df_esm_maskseq_domainome_3['id'] = df_esm_maskseq_domainome_3['code'] + '_' + df_esm_maskseq_domainome_3['mut_type']
df_esm_maskseq_domainome_3 = df_esm_maskseq_domainome_3.set_index('id')

df_esm_maskseq_domainome = pd.concat([df_esm_maskseq_domainome_1, df_esm_maskseq_domainome_2[[c for c in df_esm_maskseq_domainome_2.columns if 'MSR' in c]], df_esm_maskseq_domainome_3[[c for c in df_esm_maskseq_domainome_3.columns if 'MSR' in c]]], axis=1)

df_esm_maskseq_domainome

           ESM-MSR_maskseq_1  ESM-MSR_maskseq_additive_1
uid                                                     
1TUD_A24C          -0.000530                   -0.000530
1TUD_A24F          -0.001358                   -0.001358
1TUD_A24G          -0.000801                   -0.000801
1TUD_A24I          -0.001127                   -0.001127
1TUD_A24K          -0.001338                   -0.001338
...                      ...                         ...
2L2P_Y9R           -0.001058                   -0.001058
2L2P_Y9S           -0.000769                   -0.000769
2L2P_Y9T           -0.001231                   -0.001231
2L2P_Y9V           -0.001111                   -0.001111
2L2P_Y9W           -0.000711                   -0.000711

[190559 rows x 2 columns]


,code,mut_seq,mut_type,uniprot_ID,pdb_file,ddG_ML,chain,pdb,chain_id,wt1,pos1,mut1,delta_logit1,ddg_pred,scheme,ESM-MSR_maskseq_1,ESM-MSR_maskseq_2,ESM-MSR_maskseq_3
id,,,,,,,,,,,,,,,,,,
A0A2R8Y422_PF00240_2_A27C,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,A27C,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.025738,A,A0A2R8Y422_2-71.pdb,A,A,27,C,-5.461914,-0.000546,masked,-0.000546,-0.000546,-0.000546
A0A2R8Y422_PF00240_2_A27D,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,A27D,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.111544,A,A0A2R8Y422_2-71.pdb,A,A,27,D,-6.408203,-0.000641,masked,-0.000641,-0.000641,-0.000641
A0A2R8Y422_PF00240_2_A27E,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,A27E,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.290325,A,A0A2R8Y422_2-71.pdb,A,A,27,E,-5.153809,-0.000515,masked,-0.000515,-0.000515,-0.000515
A0A2R8Y422_PF00240_2_A27F,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,A27F,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.002225,A,A0A2R8Y422_2-71.pdb,A,A,27,F,-6.172852,-0.000617,masked,-0.000617,-0.000617,-0.000617
A0A2R8Y422_PF00240_2_A27G,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,A27G,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.033329,A,A0A2R8Y422_2-71.pdb,A,A,27,G,-5.556641,-0.000556,masked,-0.000556,-0.000556,-0.000556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6N9_PF00595_207_V83R,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83R,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.040825,A,Q9Y6N9_207-295.pdb,A,V,83,R,-1.292969,-0.000129,masked,-0.000129,-0.000129,-0.000129
Q9Y6N9_PF00595_207_V83S,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83S,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.228931,A,Q9Y6N9_207-295.pdb,A,V,83,S,-3.752930,-0.000375,masked,-0.000375,-0.000375,-0.000375
Q9Y6N9_PF00595_207_V83T,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83T,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.353383,A,Q9Y6N9_207-295.pdb,A,V,83,T,-3.219482,-0.000322,masked,-0.000322,-0.000322,-0.000322


In [6]:
df = df.join(df_esm_seqonly[[c for c in df_esm_seqonly.columns if 'MSR' in c]], how='left')
df = df.join(df_esm_maskseq[[c for c in df_esm_maskseq.columns if 'MSR' in c]], how='left')
#df = df.join(df_esm_regonly[[c for c in df_esm_regonly.columns if 'MSR' in c]], how='left')

df

,ESM3-small,ESM3-small_additive,ThermoMPNN_1,ThermoMPNN(-D)_1,ThermoMPNN_2,ThermoMPNN(-D)_2,ThermoMPNN_3,ThermoMPNN(-D)_3,ddG_ML_thermo,MutateEverything_1,...,ESM-MSR_seqonly_2,ESM-MSR_seqonly_additive_2,ESM-MSR_seqonly_3,ESM-MSR_seqonly_additive_3,ESM-MSR_maskseq_1,ESM-MSR_maskseq_additive_1,ESM-MSR_maskseq_2,ESM-MSR_maskseq_additive_2,ESM-MSR_maskseq_3,ESM-MSR_maskseq_additive_3
uid,,,,,,,,,,,,,,,,,,,,,
1TUD_A24C,-4.390625,NaN,-0.925135,-0.925135,-0.886917,-0.886917,-0.952761,-0.952761,-0.270375,-0.814228,...,-0.984433,-0.984433,-1.040257,-1.040257,-0.000530,-0.000530,-0.000530,-0.000530,-0.000530,-0.000530
1TUD_A24F,-13.187500,NaN,-1.486816,-1.486816,-1.206404,-1.206404,-1.260494,-1.260494,-3.959452,-1.489653,...,-3.703111,-3.703111,-3.710026,-3.710026,-0.001358,-0.001358,-0.001358,-0.001358,-0.001358,-0.001358
1TUD_A24G,-7.257812,NaN,-1.522053,-1.522053,-1.454631,-1.454631,-1.334050,-1.334050,-3.049759,-2.172791,...,-2.598573,-2.598573,-2.578590,-2.578590,-0.000801,-0.000801,-0.000801,-0.000801,-0.000801,-0.000801
1TUD_A24I,-9.976562,NaN,-1.099106,-1.099106,-1.049073,-1.049073,-1.186555,-1.186555,-3.423371,-0.735876,...,-3.406814,-3.406814,-3.306081,-3.306081,-0.001127,-0.001127,-0.001127,-0.001127,-0.001127,-0.001127
1TUD_A24K,-13.375000,NaN,-2.208153,-2.208153,-2.062830,-2.062830,-2.229763,-2.229763,-4.974819,-3.481638,...,-3.553149,-3.553149,-3.676468,-3.676468,-0.001338,-0.001338,-0.001338,-0.001338,-0.001338,-0.001338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2L2P_R39Y:T46R,-8.032227,-9.728516,-1.581925,-1.701419,-1.973589,-1.444623,-1.756877,-1.526226,-0.312255,-0.755020,...,-1.625480,-2.239314,-1.792003,-2.364518,-0.000891,-0.001113,-0.000891,-0.001113,-0.000891,-0.001113
2L2P_R39Y:T46S,-8.964111,-9.817871,-1.718593,-1.737093,-2.012937,-1.476977,-1.912432,-1.452884,-0.735413,-1.126285,...,-2.191657,-2.314597,-2.341913,-2.475137,-0.000895,-0.000953,-0.000895,-0.000953,-0.000895,-0.000953
2L2P_R39Y:T46V,-8.656250,-7.921875,-0.951985,-1.136660,-1.252671,-0.864598,-1.024171,-0.541343,-0.416700,-0.067048,...,-1.841523,-1.919434,-2.144135,-2.196725,-0.000977,-0.000929,-0.000977,-0.000929,-0.000977,-0.000929


In [7]:
df_domainome_1 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/domainome/msr_chain/1/epoch=08-val_rho_avg=0.758.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_domainome_1['ESM-MSR_1'] = df_domainome_1['ddg_pred']
#df_domainome_1['mut_type'] = df_domainome_1['mut_info']
df_domainome_1['id'] = df_domainome_1['code'] + '_' + df_domainome_1['mut_type']
df_domainome_1 = df_domainome_1.set_index('id')
df_domainome_1['ddG'] = df_domainome_1['ddG_ML']

df_domainome_2 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/domainome/msr_chain/2/epoch=07-val_rho_avg=0.755.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_domainome_2['ESM-MSR_2'] = df_domainome_2['ddg_pred']
df_domainome_2['mut_type'] = df_domainome_2['mut_info']
df_domainome_2['id'] = df_domainome_2['code'] + '_' + df_domainome_2['mut_type']
df_domainome_2 = df_domainome_2.set_index('id')

df_domainome_3 = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/domainome/msr_chain/3/epoch=08-val_rho_avg=0.757.ckpt_alpha12chain_rule_avg_masked.csv', index_col=0)
df_domainome_3['ESM-MSR_3'] = df_domainome_3['ddg_pred']
df_domainome_3['mut_type'] = df_domainome_3['mut_info']
df_domainome_3['id'] = df_domainome_3['code'] + '_' + df_domainome_3['mut_type']
df_domainome_3 = df_domainome_3.set_index('id')

df_domainome = pd.concat([df_domainome_1, df_domainome_2[[c for c in df_domainome_2.columns if 'MSR' in c]], df_domainome_3[[c for c in df_domainome_3.columns if 'MSR' in c]]], axis=1)

In [8]:
df_domainome = df_domainome.join(df_esm_seqonly_domainome[[c for c in df_esm_seqonly_domainome.columns if 'MSR' in c]], how='left')
df_domainome = df_domainome.join(df_esm_maskseq_domainome[[c for c in df_esm_maskseq_domainome.columns if 'MSR' in c]], how='left')
#df_domainome = df_domainome.join(df_esm_regonly_domainome[[c for c in df_esm_regonly_domainome.columns if 'MSR' in c]], how='left')

df_domainome

,code,mut_seq,mut_type,uniprot_ID,pdb_file,ddG_ML,chain,pdb,chain_id,wt1,...,ESM-MSR_1,ddG,ESM-MSR_2,ESM-MSR_3,ESM-MSR_seqonly_1,ESM-MSR_seqonly_2,ESM-MSR_seqonly_3,ESM-MSR_maskseq_1,ESM-MSR_maskseq_2,ESM-MSR_maskseq_3
id,,,,,,,,,,,,,,,,,,,,,
A0A2R8Y422_PF00240_2_A27C,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKCKIQDKEGIPPDQQRLIFAG...,A27C,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.025738,A,A0A2R8Y422_2-71.pdb,A,A,...,-0.056039,0.025738,-0.157327,-0.070897,-0.252836,-0.252836,-0.273149,-0.000546,-0.000546,-0.000546
A0A2R8Y422_PF00240_2_A27D,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKDKIQDKEGIPPDQQRLIFAG...,A27D,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.111544,A,A0A2R8Y422_2-71.pdb,A,A,...,-1.103929,0.111544,-1.324483,-1.225101,-1.180352,-1.180352,-1.202612,-0.000641,-0.000641,-0.000641
A0A2R8Y422_PF00240_2_A27E,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKEKIQDKEGIPPDQQRLIFAG...,A27E,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.290325,A,A0A2R8Y422_2-71.pdb,A,A,...,-0.670991,0.290325,-0.882019,-0.783867,-0.632656,-0.632656,-0.707002,-0.000515,-0.000515,-0.000515
A0A2R8Y422_PF00240_2_A27F,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKFKIQDKEGIPPDQQRLIFAG...,A27F,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.002225,A,A0A2R8Y422_2-71.pdb,A,A,...,-0.479389,0.002225,-0.615267,-0.498679,-0.270032,-0.270032,-0.337081,-0.000617,-0.000617,-0.000617
A0A2R8Y422_PF00240_2_A27G,A0A2R8Y422_PF00240_2,QIFVKTLMGKTITLEVELSDTIDNVKGKIQDKEGIPPDQQRLIFAG...,A27G,A0A2R8Y422,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.033329,A,A0A2R8Y422_2-71.pdb,A,A,...,-0.829185,-0.033329,-0.984897,-0.881620,-1.081788,-1.081788,-1.100693,-0.000556,-0.000556,-0.000556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6N9_PF00595_207_V83R,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83R,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,0.040825,A,Q9Y6N9_207-295.pdb,A,V,...,-0.330782,0.040825,-0.318167,-0.358626,-0.734395,-0.734395,-0.749727,-0.000129,-0.000129,-0.000129
Q9Y6N9_PF00595_207_V83S,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83S,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.228931,A,Q9Y6N9_207-295.pdb,A,V,...,-0.735856,-0.228931,-0.646069,-0.738129,-1.592447,-1.592447,-1.638096,-0.000375,-0.000375,-0.000375
Q9Y6N9_PF00595_207_V83T,Q9Y6N9_PF00595_207,NKEKKVFISLVGSRGLGCSISSGPIQKPGIFISHVKPGSLSAEVGL...,V83T,Q9Y6N9,/home/sareeves/PSLMs/data/domainome1/alphafold...,-0.353383,A,Q9Y6N9_207-295.pdb,A,V,...,-0.586107,-0.353383,-0.536838,-0.599496,-1.148795,-1.148795,-1.185075,-0.000322,-0.000322,-0.000322


In [9]:
# Get all grouped and ungrouped ranking correlations

df_scores_domainome = pd.read_csv('~/PSLMs/data/lora/test_scores_final_spearman.csv', index_col=[0,1])

print('ESM-MSR')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_1', 'Domainome1', pred_col='ESM-MSR_1', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_2', 'Domainome1', pred_col='ESM-MSR_2', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_3', 'Domainome1', pred_col='ESM-MSR_3', label='ddG')
df_scores_domainome = df_scores_domainome.iloc[:, -6:].dropna(how='all')

print('ESM-MSR seqonly')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_seqonly_1', 'Domainome1', pred_col='ESM-MSR_seqonly_1', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_seqonly_2', 'Domainome1', pred_col='ESM-MSR_seqonly_2', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_seqonly_3', 'Domainome1', pred_col='ESM-MSR_seqonly_3', label='ddG')

print('ESM-MSR_maskseq')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_maskseq_1', 'Domainome1', pred_col='ESM-MSR_maskseq_1', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_maskseq_2', 'Domainome1', pred_col='ESM-MSR_maskseq_2', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM-MSR_maskseq_3', 'Domainome1', pred_col='ESM-MSR_maskseq_3', label='ddG')


"""
print('ThermoMPNN')
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ThermoMPNN(-D)_1', 'Domainome1', pred_col='ThermoMPNN(-D)_1', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ThermoMPNN(-D)_2', 'Domainome1', pred_col='ThermoMPNN(-D)_2', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ThermoMPNN(-D)_3', 'Domainome1', pred_col='ThermoMPNN(-D)_3', labe
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'MutateEverything_1', 'Domainome1',
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'MutateEverything_2', 'Domainome1',
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'MutateEverything_3', 'Domainome1', pred_col='MutateEverything_3', label='ddG')

print('ESM3-small-open')
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM3-small-open', 'Domainome1', pred_col='ESM3-small-open', label='ddG')
print('ESM3-small')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM3-small', 'Domainome1', pred_col='ESM3-small', label='ddG')
print('ESM3-medium')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM3-medium', 'Domainome1', pred_col='ESM3-medium', label='ddG')
print('ESM3-large')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ESM3-large', 'Domainome1', pred_col='ESM3-large', label='ddG')
print('Rosetta')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'Rosetta Cartesian DDG_1', 'Domainome1', pred_col='Rosetta Cartesian DDG_1', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'Rosetta Cartesian DDG_2', 'Domainome1', pred_col='Rosetta Cartesian DDG_2', label='ddG')
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'Rosetta Cartesian DDG_3', 'Domainome1', pred_col='Rosetta Cartesian DDG_3',
df_scores_domainome = assess_grouped_spearman(df_domainome, df_scores_domainome, 'ProteinMPNN', 'Domainome1', pred_col='ProteinMPNN', label='ddG')
"""

df_scores_domainome

ESM-MSR
ESM-MSR seqonly
ESM-MSR_maskseq


ESM-MSR_1  ESM-MSR_1_n  ESM-MSR_2  ESM-MSR_2_n  \
dataset    grouping                                                    
Domainome1 ungrouped   0.525775     514596.0   0.517903     514596.0   
           grouped     0.539041     514596.0   0.535167     514596.0   

                      ESM-MSR_3  ESM-MSR_3_n  ESM-MSR_seqonly_1  \
dataset    grouping                                               
Domainome1 ungrouped   0.525617     514596.0           0.527544   
           grouped     0.537655     514596.0           0.534304   

                      ESM-MSR_seqonly_1_n  ESM-MSR_seqonly_2  \
dataset    grouping                                            
Domainome1 ungrouped             514596.0           0.527544   
           grouped               514596.0           0.534304   

                      ESM-MSR_seqonly_2_n  ESM-MSR_seqonly_3  \
dataset    grouping                                            
Domainome1 ungrouped             514596.0           0.532094   
           grouped               514596.0           0.539167   

                      ESM-MSR_seqonly_3_n  ESM-MSR_maskseq_1  \
dataset    grouping                                            
Domainome1 ungrouped             514596.0           0.530248   
           grouped               514596.0           0.553908   

                      ESM-MSR_maskseq_1_n  ESM-MSR_maskseq_2  \
dataset    grouping                                            
Domainome1 ungrouped             514596.0           0.530248   
           grouped               514596.0           0.553908   

                      ESM-MSR_maskseq_2_n  ESM-MSR_maskseq_3  \
dataset    grouping                                            
Domainome1 ungrouped             514596.0           0.530248   
           grouped               514596.0           0.553908   

                      ESM-MSR_maskseq_3_n  
dataset    grouping                        
Domainome1 ungrouped             514596.0  
           grouped               514596.0

In [10]:
# Get all grouped and ungrouped ranking correlations

df_scores = pd.read_csv('~/PSLMs/data/lora/test_scores_final_spearman.csv', index_col=[0,1])

print('ESM-MSR')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_1', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_1', 'feb19_tsuboyama-test', pred_col='ESM-MSR_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_2', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_2', 'feb19_tsuboyama-test', pred_col='ESM-MSR_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_3', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_3', 'feb19_tsuboyama-test', pred_col='ESM-MSR_3', label='ddG')
df_scores = df_scores.iloc[:, -6:].dropna(how='all')

print('ESM-MSR seqonly')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_1', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_seqonly_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_seqonly_1', 'feb19_tsuboyama-test', pred_col='ESM-MSR_seqonly_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_2', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_seqonly_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_seqonly_2', 'feb19_tsuboyama-test', pred_col='ESM-MSR_seqonly_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_3', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_seqonly_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_seqonly_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_seqonly_3', 'feb19_tsuboyama-test', pred_col='ESM-MSR_seqonly_3', label='ddG')

print('ESM-MSR_maskseq')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_1', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_maskseq_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_maskseq_1', 'feb19_tsuboyama-test', pred_col='ESM-MSR_maskseq_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_2', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_maskseq_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_maskseq_2', 'feb19_tsuboyama-test', pred_col='ESM-MSR_maskseq_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_3', 'feb19_tsuboyama-S-test', pred_col='ESM-MSR_maskseq_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM-MSR_maskseq_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM-MSR_maskseq_3', 'feb19_tsuboyama-test', pred_col='ESM-MSR_maskseq_3', label='ddG')

print('ThermoMPNN')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_1', 'feb19_tsuboyama-S-test', pred_col='ThermoMPNN(-D)_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_1', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ThermoMPNN(-D)_1', 'feb19_tsuboyama-test', pred_col='ThermoMPNN(-D)_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_2', 'feb19_tsuboyama-S-test', pred_col='ThermoMPNN(-D)_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_2', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ThermoMPNN(-D)_2', 'feb19_tsuboyama-test', pred_col='ThermoMPNN(-D)_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_3', 'feb19_tsuboyama-S-test', pred_col='ThermoMPNN(-D)_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ThermoMPNN(-D)_3', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ThermoMPNN(-D)_3', 'feb19_tsuboyama-test', pred_col='ThermoMPNN(-D)_3', label='ddG')

print('MutateEverything')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_1', 'feb19_tsuboyama-S-test', pred_col='MutateEverything_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_1', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'MutateEverything_1', 'feb19_tsuboyama-test', pred_col='MutateEverything_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_2', 'feb19_tsuboyama-S-test', pred_col='MutateEverything_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_2', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'MutateEverything_2', 'feb19_tsuboyama-test', pred_col='MutateEverything_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_3', 'feb19_tsuboyama-S-test', pred_col='MutateEverything_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'MutateEverything_3', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'MutateEverything_3', 'feb19_tsuboyama-test', pred_col='MutateEverything_3', label='ddG')

print('ESM3-small-open')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM3-small-open', 'feb19_tsuboyama-S-test', pred_col='ESM3-small-open', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM3-small-open', 'feb19_tsuboyama-D-test', pred_col='ESM3-small-open', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM3-small-open', 'feb19_tsuboyama-test', pred_col='ESM3-small-open', label='ddG')
print('ESM3-small')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM3-small', 'feb19_tsuboyama-S-test', pred_col='ESM3-small', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM3-small', 'feb19_tsuboyama-D-test', pred_col='ESM3-small', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM3-small', 'feb19_tsuboyama-test', pred_col='ESM3-small', label='ddG')
print('ESM3-medium')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM3-medium', 'feb19_tsuboyama-S-test', pred_col='ESM3-medium', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM3-medium', 'feb19_tsuboyama-D-test', pred_col='ESM3-medium', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM3-medium', 'feb19_tsuboyama-test', pred_col='ESM3-medium', label='ddG')
print('ESM3-large')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ESM3-large', 'feb19_tsuboyama-S-test', pred_col='ESM3-large', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ESM3-large', 'feb19_tsuboyama-D-test', pred_col='ESM3-large', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ESM3-large', 'feb19_tsuboyama-test', pred_col='ESM3-large', label='ddG')
print('Rosetta')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_1', 'feb19_tsuboyama-S-test', pred_col='Rosetta Cartesian DDG_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_1', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_1', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'Rosetta Cartesian DDG_1', 'feb19_tsuboyama-test', pred_col='Rosetta Cartesian DDG_1', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_2', 'feb19_tsuboyama-S-test', pred_col='Rosetta Cartesian DDG_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_2', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_2', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'Rosetta Cartesian DDG_2', 'feb19_tsuboyama-test', pred_col='Rosetta Cartesian DDG_2', label='ddG')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_3', 'feb19_tsuboyama-S-test', pred_col='Rosetta Cartesian DDG_3', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'Rosetta Cartesian DDG_3', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_3', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'Rosetta Cartesian DDG_3', 'feb19_tsuboyama-test', pred_col='Rosetta Cartesian DDG_3', label='ddG')
print('ProteinMPNN')
df_scores = assess_grouped_spearman(df.loc[~df['mut_type'].str.contains(':')], df_scores, 'ProteinMPNN', 'feb19_tsuboyama-S-test', pred_col='ProteinMPNN', label='ddG')
df_scores = assess_grouped_spearman(df.loc[df['mut_type'].str.contains(':')], df_scores, 'ProteinMPNN', 'feb19_tsuboyama-D-test', pred_col='ProteinMPNN', label='ddG')
df_scores = assess_grouped_spearman(df, df_scores, 'ProteinMPNN', 'feb19_tsuboyama-test', pred_col='ProteinMPNN', label='ddG')

df_scores

ESM-MSR
ESM-MSR seqonly
ESM-MSR_maskseq
ThermoMPNN
MutateEverything
ESM3-small-open
ESM3-small
ESM3-medium
ESM3-large
Rosetta
ProteinMPNN


ESM-MSR_1  ESM-MSR_1_n  ESM-MSR_2  \
dataset                grouping                                       
feb19_tsuboyama-test   ungrouped   0.771120     190559.0   0.762400   
                       grouped     0.814266     190559.0   0.807293   
feb19_tsuboyama-S-test ungrouped   0.768254     135901.0   0.756956   
                       grouped     0.821445     135901.0   0.814888   
feb19_tsuboyama-D-test ungrouped   0.591349      54658.0   0.583896   
                       grouped     0.681614      54658.0   0.671595   

                                  ESM-MSR_2_n  ESM-MSR_3  ESM-MSR_3_n  \
dataset                grouping                                         
feb19_tsuboyama-test   ungrouped     190559.0   0.769626     190559.0   
                       grouped       190559.0   0.815254     190559.0   
feb19_tsuboyama-S-test ungrouped     135901.0   0.768503     135901.0   
                       grouped       135901.0   0.822569     135901.0   
feb19_tsuboyama-D-test ungrouped      54658.0   0.583727      54658.0   
                       grouped        54658.0   0.679438      54658.0   

                                  ESM-MSR_seqonly_1  ESM-MSR_seqonly_1_n  \
dataset                grouping                                            
feb19_tsuboyama-test   ungrouped           0.746953             190559.0   
                       grouped             0.773764             190559.0   
feb19_tsuboyama-S-test ungrouped           0.736277             135901.0   
                       grouped             0.777455             135901.0   
feb19_tsuboyama-D-test ungrouped           0.541502              54658.0   
                       grouped             0.615006              54658.0   

                                  ESM-MSR_seqonly_2  ESM-MSR_seqonly_2_n  ...  \
dataset                grouping                                           ...   
feb19_tsuboyama-test   ungrouped           0.746953             190559.0  ...   
                       grouped             0.773764             190559.0  ...   
feb19_tsuboyama-S-test ungrouped           0.736277             135901.0  ...   
                       grouped             0.777455             135901.0  ...   
feb19_tsuboyama-D-test ungrouped           0.541502              54658.0  ...   
                       grouped             0.615006              54658.0  ...   

                                  ESM3-large  ESM3-large_n  \
dataset                grouping                              
feb19_tsuboyama-test   ungrouped    0.551599      190559.0   
                       grouped      0.557471      190559.0   
feb19_tsuboyama-S-test ungrouped    0.462200      135901.0   
                       grouped      0.541152      135901.0   
feb19_tsuboyama-D-test ungrouped    0.340445       54658.0   
                       grouped      0.339908       54658.0   

                                  Rosetta Cartesian DDG_1  \
dataset                grouping                             
feb19_tsuboyama-test   ungrouped                 0.598006   
                       grouped                   0.613936   
feb19_tsuboyama-S-test ungrouped                 0.587846   
                       grouped                   0.601113   
feb19_tsuboyama-D-test ungrouped                 0.445305   
                       grouped                   0.462961   

                                  Rosetta Cartesian DDG_1_n  \
dataset                grouping                               
feb19_tsuboyama-test   ungrouped                   190557.0   
                       grouped                     190557.0   
feb19_tsuboyama-S-test ungrouped                   135899.0   
                       grouped                     135899.0   
feb19_tsuboyama-D-test ungrouped                    54658.0   
                       grouped                      54658.0   

                                  Rosetta Cartesian DDG_2  \
dataset                grouping                             
feb19_tsu

In [11]:
df2 = df_scores[[c for c in df_scores.columns if not c.endswith('_n')]].T
df2

dataset                 feb19_tsuboyama-test           feb19_tsuboyama-S-test  \
grouping                           ungrouped   grouped              ungrouped   
ESM-MSR_1                           0.771120  0.814266               0.768254   
ESM-MSR_2                           0.762400  0.807293               0.756956   
ESM-MSR_3                           0.769626  0.815254               0.768503   
ESM-MSR_seqonly_1                   0.746953  0.773764               0.736277   
ESM-MSR_seqonly_2                   0.746953  0.773764               0.736277   
ESM-MSR_seqonly_3                   0.742958  0.766822               0.731192   
ESM-MSR_maskseq_1                   0.618674  0.623386               0.561232   
ESM-MSR_maskseq_2                   0.618674  0.623386               0.561232   
ESM-MSR_maskseq_3                   0.618674  0.623386               0.561232   
ThermoMPNN(-D)_1                    0.738021  0.762149               0.741463   
ThermoMPNN(-D)_2                    0.744475  0.764392               0.740680   
ThermoMPNN(-D)_3                    0.739281  0.765412               0.742179   
MutateEverything_1                  0.716863  0.751450               0.738291   
MutateEverything_2                  0.698163  0.744652               0.732659   
MutateEverything_3                  0.707226  0.749257               0.735624   
ESM3-small-open                     0.618674  0.623386               0.561232   
ESM3-small                          0.626997  0.633564               0.574919   
ESM3-medium                         0.630647  0.635058               0.577153   
ESM3-large                          0.551599  0.557471               0.462200   
Rosetta Cartesian DDG_1             0.598006  0.613936               0.587846   
Rosetta Cartesian DDG_2             0.598380  0.614454               0.588167   
Rosetta Cartesian DDG_3             0.598368  0.614603               0.587788   
ProteinMPNN                         0.627759  0.613430               0.590528   

dataset                           feb19_tsuboyama-D-test            
grouping                  grouped              ungrouped   grouped  
ESM-MSR_1                0.821445               0.591349  0.681614  
ESM-MSR_2                0.814888               0.583896  0.671595  
ESM-MSR_3                0.822569               0.583727  0.679438  
ESM-MSR_seqonly_1        0.777455               0.541502  0.615006  
ESM-MSR_seqonly_2        0.777455               0.541502  0.615006  
ESM-MSR_seqonly_3        0.770189               0.528669  0.601278  
ESM-MSR_maskseq_1        0.614174               0.432999  0.417111  
ESM-MSR_maskseq_2        0.614174               0.432999  0.417111  
ESM-MSR_maskseq_3        0.614174               0.432999  0.417111  
ThermoMPNN(-D)_1         0.772760               0.424529  0.569891  
ThermoMPNN(-D)_2         0.772195               0.467111  0.572193  
ThermoMPNN(-D)_3         0.774492               0.460441  0.568898  
MutateEverything_1       0.765411               0.491115  0.584362  
MutateEverything_2       0.763605               0.438339  0.572598  
MutateEverything_3       0.766504               0.449201  0.573760  
ESM3-small-open          0.614174               0.432999  0.417111  
ESM3-small               0.624249               0.409270  0.426438  
ESM3-medium              0.625658               0.406615  0.412495  
ESM3-large               0.541152               0.340445  0.339908  
Rosetta Cartesian DDG_1  0.601113               0.445305  0.462961  
Rosetta Cartesian DDG_2  0.601386               0.445584  0.462114  
Rosetta Cartesian DDG_3  0.601339               0.445463  0.464262  
ProteinMPNN              0.613229               0.379038  0.367683

In [12]:
df2_domainome = df_scores_domainome[[c for c in df_scores_domainome.columns if not c.endswith('_n')]].T
df2_domainome

dataset           Domainome1          
grouping           ungrouped   grouped
ESM-MSR_1           0.525775  0.539041
ESM-MSR_2           0.517903  0.535167
ESM-MSR_3           0.525617  0.537655
ESM-MSR_seqonly_1   0.527544  0.534304
ESM-MSR_seqonly_2   0.527544  0.534304
ESM-MSR_seqonly_3   0.532094  0.539167
ESM-MSR_maskseq_1   0.530248  0.553908
ESM-MSR_maskseq_2   0.530248  0.553908
ESM-MSR_maskseq_3   0.530248  0.553908

In [13]:
import pandas as pd
import numpy as np
import re

def collapse_replicates(df):
    """
    Takes a dataframe where the index contains replicates with suffixes (e.g., _1, _2).
    Groups them by the base name and replaces the values with 'mean ± std'.
    """
    # Ensure the input is a copy to avoid SettingWithCopy warnings on the original
    df = df.copy()

    # 1. Identify the grouping key (Base Name)
    # We use regex to remove '_\d+' (underscore followed by digits) from the end of the string
    # If the index is not currently set, we assume the first column is the identifier
    if df.index.name is None and not df.index.is_object():
         df = df.set_index(df.columns[0])
    
    # Extract base names (e.g., "ESM-MSR_1" -> "ESM-MSR")
    base_names = df.index.astype(str).str.replace(r'_\d+$', '', regex=True)

    # 2. Group by the base name and calculate mean and std
    grouped = df.groupby(base_names)
    means = grouped.mean()
    stds = grouped.std()

    # 3. Format the output columns
    # We create a new DataFrame to store the string results
    result_df = pd.DataFrame(index=means.index, columns=means.columns)

    # Iterate through columns to format as "mean ± std"
    for col in means.columns:
        m = means[col]
        s = stds[col]
        
        # Combine mean and std. 
        # We fill NaN stds with 0 (happens if there was only 1 replicate)
        # You can adjust the {:.6f} to change the decimal precision
        result_df[col] = [
            f"{val_m:.3f} ± {val_s:.3f}" if not np.isnan(val_s) else f"{val_m:.6f}"
            for val_m, val_s in zip(m, s)
        ]

    return result_df

processed_df = collapse_replicates(df2)
print(processed_df)

processed_df_domainome = collapse_replicates(df2_domainome)
print(processed_df_domainome)

dataset               feb19_tsuboyama-test                 \
grouping                         ungrouped        grouped   
ESM-MSR                      0.768 ± 0.005  0.812 ± 0.004   
ESM-MSR_maskseq              0.619 ± 0.000  0.623 ± 0.000   
ESM-MSR_seqonly              0.746 ± 0.002  0.771 ± 0.004   
ESM3-large                        0.551599       0.557471   
ESM3-medium                       0.630647       0.635058   
ESM3-small                        0.626997       0.633564   
ESM3-small-open                   0.618674       0.623386   
MutateEverything             0.707 ± 0.009  0.748 ± 0.003   
ProteinMPNN                       0.627759       0.613430   
Rosetta Cartesian DDG        0.598 ± 0.000  0.614 ± 0.000   
ThermoMPNN(-D)               0.741 ± 0.003  0.764 ± 0.002   

dataset               feb19_tsuboyama-S-test                 \
grouping                           ungrouped        grouped   
ESM-MSR                        0.765 ± 0.007  0.820 ± 0.004   
ESM-MSR_maskseq  

In [14]:
# add additive scores
# should be 4343 mutants missing for each method
from analysis_utils_msr import sum_individual_mutation_scores, compute_dddg

cols = df.columns.drop('code').drop('mut_type')
cols = ['Rosetta Cartesian DDG_1', 'Rosetta Cartesian DDG_2', 'Rosetta Cartesian DDG_3', 'ddG_ML_me']
df_epistatic = df.copy(deep=True)

#print(len(df.loc[df['mut_type'].str.contains(':')]))
for c in cols:
    if not c.endswith('_n'):
        print(c)
        # synthesize additive mutation predictions and scores
        df_epistatic, _, _, _ = sum_individual_mutation_scores(df_epistatic, c, new_score_column=None)

#df_epistatic = df_epistatic[[c for c in df_epistatic.columns if 'additive' in c] + ['code', 'ddG', 'mut_type']]
df_epistatic = df_epistatic[[c for c in df_epistatic.columns if not c.endswith('_n')]]
df_epistatic = df_epistatic.dropna(subset='ESM-MSR_1')
df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN(-D)_1_additive'] = df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN_1']
df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN(-D)_2_additive'] = df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN_2']
df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN(-D)_3_additive'] = df_epistatic.loc[df_epistatic.index.str.contains(':'), 'ThermoMPNN_3']

print(len(df_epistatic))
df_epistatic['ThermoMPNN(-D)_1_additive'].dropna()

df_scores_dddG = pd.read_csv('~/PSLMs/data/lora/test_scores_new.csv', index_col=[0,1])
df_dddG = df_epistatic.copy(deep=True)
df_dddG = compute_dddg(df_dddG)
df_dddG = df_dddG[[c for c in df_dddG.columns if 'dddG' in c]+['code', 'mut_type', 'ddG']]
df_dddG.columns = [c[:-5] if c.endswith('dddG') else c for c in df_dddG.columns ]
df_dddG.columns

Rosetta Cartesian DDG_1
Rosetta Cartesian DDG_2
Rosetta Cartesian DDG_3
ddG_ML_me
190559


Index(['ESM3-small', 'MutateEverything_1', 'MutateEverything_2',
       'MutateEverything_3', 'ESM3-medium', 'ESM3-large', 'ProteinMPNN',
       'ESM-MSR_1', 'ESM-MSR_2', 'ESM-MSR_3', 'ESM3-small-open',
       'ESM-MSR_seqonly_1', 'ESM-MSR_seqonly_2', 'ESM-MSR_seqonly_3',
       'ESM-MSR_maskseq_1', 'ESM-MSR_maskseq_2', 'ESM-MSR_maskseq_3',
       'Rosetta Cartesian DDG_1', 'Rosetta Cartesian DDG_2',
       'Rosetta Cartesian DDG_3', 'ddG_ML_me', 'ThermoMPNN(-D)_1',
       'ThermoMPNN(-D)_2', 'ThermoMPNN(-D)_3', 'code', 'mut_type', 'ddG'],
      dtype='object')

In [15]:
df_dddG['ESM-MSR_1']

uid
1TUD_A24C         0.000000
1TUD_A24F         0.000000
1TUD_A24G         0.000000
1TUD_A24I         0.000000
1TUD_A24K         0.000000
                    ...   
2L2P_R39Y:T46R    0.576040
2L2P_R39Y:T46S    0.057787
2L2P_R39Y:T46V   -0.086962
2L2P_R39Y:T46W    0.218192
2L2P_R39Y:T46Y    0.247835
Name: ESM-MSR_1, Length: 190559, dtype: float64

In [16]:
# Get all grouped and ungrouped ranking correlations
df_dddG_scores = pd.read_csv('~/PSLMs/data/lora/test_scores_new.csv', index_col=[0,1])

print('ESM-MSR')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_3', label='ddG_ML_me')
df_dddG_scores = df_dddG_scores.iloc[:, -6:].dropna(how='all')

print('ESM-MSR_seqonly')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_seqonly_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_seqonly_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_seqonly_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_seqonly_3', label='ddG_ML_me')

print('ESM-MSR_maskseq')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_maskseq_1', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_maskseq_2', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM-MSR_maskseq_3', 'feb19_tsuboyama-D-test', pred_col='ESM-MSR_maskseq_3', label='ddG_ML_me')

print('ThermoMPNN')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ThermoMPNN(-D)_1', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ThermoMPNN(-D)_2', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ThermoMPNN(-D)_3', 'feb19_tsuboyama-D-test', pred_col='ThermoMPNN(-D)_3', label='ddG_ML_me')

print('MutateEverything')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'MutateEverything_1', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'MutateEverything_2', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'MutateEverything_3', 'feb19_tsuboyama-D-test', pred_col='MutateEverything_3', label='ddG_ML_me')

print('ESM3-small-open')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM3-small-open', 'feb19_tsuboyama-D-test', pred_col='ESM3-small-open', label='ddG_ML_me')
print('ESM3-small')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM3-small', 'feb19_tsuboyama-D-test', pred_col='ESM3-small', label='ddG_ML_me')
print('ESM3-medium')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM3-medium', 'feb19_tsuboyama-D-test', pred_col='ESM3-medium', label='ddG_ML_me')
print('ESM3-large')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ESM3-large', 'feb19_tsuboyama-D-test', pred_col='ESM3-large', label='ddG_ML_me')
print('Rosetta')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'Rosetta Cartesian DDG_1', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_1', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'Rosetta Cartesian DDG_2', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_2', label='ddG_ML_me')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'Rosetta Cartesian DDG_3', 'feb19_tsuboyama-D-test', pred_col='Rosetta Cartesian DDG_3', label='ddG_ML_me')
print('ProteinMPNN')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ProteinMPNN', 'feb19_tsuboyama-D-test', pred_col='ProteinMPNN', label='ddG_ML_me')
print('DDG')
df_dddG_scores = assess_grouped_spearman(df_dddG.loc[df_dddG['mut_type'].str.contains(':')], df_dddG_scores, 'ddG', 'feb19_tsuboyama-D-test', pred_col='ddG', label='ddG_ML_me')

df_dddG_scores

ESM-MSR
ESM-MSR_seqonly
ESM-MSR_maskseq
ThermoMPNN
MutateEverything
ESM3-small-open
ESM3-small
ESM3-medium
ESM3-large
Rosetta
ProteinMPNN
DDG


ESM-MSR_1  ESM-MSR_1_n  ESM-MSR_2  \
dataset                grouping                                       
feb19_tsuboyama-D-test ungrouped   0.548047      50315.0   0.546990   
                       grouped     0.519008      50315.0   0.527433   

                                  ESM-MSR_2_n  ESM-MSR_3  ESM-MSR_3_n  \
dataset                grouping                                         
feb19_tsuboyama-D-test ungrouped      50315.0   0.541505      50315.0   
                       grouped        50315.0   0.516355      50315.0   

                                  ESM-MSR_seqonly_1  ESM-MSR_seqonly_1_n  \
dataset                grouping                                            
feb19_tsuboyama-D-test ungrouped           0.506740              50315.0   
                       grouped             0.471997              50315.0   

                                  ESM-MSR_seqonly_2  ESM-MSR_seqonly_2_n  ...  \
dataset                grouping                                           ...   
feb19_tsuboyama-D-test ungrouped           0.506740              50315.0  ...   
                       grouped             0.471997              50315.0  ...   

                                  Rosetta Cartesian DDG_1  \
dataset                grouping                             
feb19_tsuboyama-D-test ungrouped                 0.194960   
                       grouped                   0.169587   

                                  Rosetta Cartesian DDG_1_n  \
dataset                grouping                               
feb19_tsuboyama-D-test ungrouped                    50315.0   
                       grouped                      50315.0   

                                  Rosetta Cartesian DDG_2  \
dataset                grouping                             
feb19_tsuboyama-D-test ungrouped                 0.203878   
                       grouped                   0.173927   

                                  Rosetta Cartesian DDG_2_n  \
dataset                grouping                               
feb19_tsuboyama-D-test ungrouped                    50315.0   
                       grouped                      50315.0   

                                  Rosetta Cartesian DDG_3  \
dataset                grouping                             
feb19_tsuboyama-D-test ungrouped                 0.206824   
                       grouped                   0.174561   

                                  Rosetta Cartesian DDG_3_n  ProteinMPNN  \
dataset                grouping                                            
feb19_tsuboyama-D-test ungrouped                    50315.0     0.332573   
                       grouped                      50315.0     0.286193   

                                  ProteinMPNN_n       ddG    ddG_n  
dataset                grouping                                     
feb19_tsuboyama-D-test ungrouped        50315.0 -0.165699  50315.0  
                       grouped          50315.0 -0.057545  50315.0  

[2 rows x 48 columns]

In [17]:
df3 = df_dddG_scores[[c for c in df_dddG_scores.columns if not c.endswith('_n')]].T
df3.columns = pd.MultiIndex.from_tuples([('feb19_tsuboyama-D-test-dddG', 'ungrouped'), ('feb19_tsuboyama-D-test-dddG', 'grouped')])
df3

feb19_tsuboyama-D-test-dddG          
                                          ungrouped   grouped
ESM-MSR_1                                  0.548047  0.519008
ESM-MSR_2                                  0.546990  0.527433
ESM-MSR_3                                  0.541505  0.516355
ESM-MSR_seqonly_1                          0.506740  0.471997
ESM-MSR_seqonly_2                          0.506740  0.471997
ESM-MSR_seqonly_3                          0.491440  0.453956
ESM-MSR_maskseq_1                          0.449304  0.432372
ESM-MSR_maskseq_2                          0.449304  0.432372
ESM-MSR_maskseq_3                          0.449304  0.432372
ThermoMPNN(-D)_1                           0.197232  0.256566
ThermoMPNN(-D)_2                           0.186109  0.231656
ThermoMPNN(-D)_3                           0.140153  0.259484
MutateEverything_1                         0.425313  0.415420
MutateEverything_2                         0.412693  0.415264
MutateEverything_3                         0.437399  0.412944
ESM3-small-open                            0.449304  0.432369
ESM3-small                                 0.441203  0.422059
ESM3-medium                                0.435070  0.405700
ESM3-large                                 0.303242  0.353427
Rosetta Cartesian DDG_1                    0.194960  0.169587
Rosetta Cartesian DDG_2                    0.203878  0.173927
Rosetta Cartesian DDG_3                    0.206824  0.174561
ProteinMPNN                                0.332573  0.286193
ddG                                       -0.165699 -0.057545

In [18]:
df3 = df2.join(df3)
processed_df = collapse_replicates(df3)
print(processed_df)

                      feb19_tsuboyama-test                 \
                                 ungrouped        grouped   
ESM-MSR                      0.768 ± 0.005  0.812 ± 0.004   
ESM-MSR_maskseq              0.619 ± 0.000  0.623 ± 0.000   
ESM-MSR_seqonly              0.746 ± 0.002  0.771 ± 0.004   
ESM3-large                        0.551599       0.557471   
ESM3-medium                       0.630647       0.635058   
ESM3-small                        0.626997       0.633564   
ESM3-small-open                   0.618674       0.623386   
MutateEverything             0.707 ± 0.009  0.748 ± 0.003   
ProteinMPNN                       0.627759       0.613430   
Rosetta Cartesian DDG        0.598 ± 0.000  0.614 ± 0.000   
ThermoMPNN(-D)               0.741 ± 0.003  0.764 ± 0.002   

                      feb19_tsuboyama-S-test                 \
                                   ungrouped        grouped   
ESM-MSR                        0.765 ± 0.007  0.820 ± 0.004   
ESM-MSR_maskseq  

In [19]:
processed_df = processed_df.join(processed_df_domainome, how='outer')


In [20]:
processed_df

feb19_tsuboyama-test                 \
                                 ungrouped        grouped   
ESM-MSR                      0.768 ± 0.005  0.812 ± 0.004   
ESM-MSR_maskseq              0.619 ± 0.000  0.623 ± 0.000   
ESM-MSR_seqonly              0.746 ± 0.002  0.771 ± 0.004   
ESM3-large                        0.551599       0.557471   
ESM3-medium                       0.630647       0.635058   
ESM3-small                        0.626997       0.633564   
ESM3-small-open                   0.618674       0.623386   
MutateEverything             0.707 ± 0.009  0.748 ± 0.003   
ProteinMPNN                       0.627759       0.613430   
Rosetta Cartesian DDG        0.598 ± 0.000  0.614 ± 0.000   
ThermoMPNN(-D)               0.741 ± 0.003  0.764 ± 0.002   

                      feb19_tsuboyama-S-test                 \
                                   ungrouped        grouped   
ESM-MSR                        0.765 ± 0.007  0.820 ± 0.004   
ESM-MSR_maskseq                0.561 ± 0.000  0.614 ± 0.000   
ESM-MSR_seqonly                0.735 ± 0.003  0.775 ± 0.004   
ESM3-large                          0.462200       0.541152   
ESM3-medium                         0.577153       0.625658   
ESM3-small                          0.574919       0.624249   
ESM3-small-open                     0.561232       0.614174   
MutateEverything               0.736 ± 0.003  0.765 ± 0.001   
ProteinMPNN                         0.590528       0.613229   
Rosetta Cartesian DDG          0.588 ± 0.000  0.601 ± 0.000   
ThermoMPNN(-D)                 0.741 ± 0.001  0.773 ± 0.001   

                      feb19_tsuboyama-D-test                 \
                                   ungrouped        grouped   
ESM-MSR                        0.586 ± 0.004  0.678 ± 0.005   
ESM-MSR_maskseq                0.433 ± 0.000  0.417 ± 0.000   
ESM-MSR_seqonly                0.537 ± 0.007  0.610 ± 0.008   
ESM3-large                          0.340445       0.339908   
ESM3-medium                         0.406615       0.412495   
ESM3-small                          0.409270       0.426438   
ESM3-small-open                     0.432999       0.417111   
MutateEverything               0.460 ± 0.028  0.577 ± 0.006   
ProteinMPNN                         0.379038       0.367683   
Rosetta Cartesian DDG          0.445 ± 0.000  0.463 ± 0.001   
ThermoMPNN(-D)                 0.451 ± 0.023  0.570 ± 0.002   

                      feb19_tsuboyama-D-test-dddG                 \
                                        ungrouped        grouped   
ESM-MSR                             0.546 ± 0.004  0.521 ± 0.006   
ESM-MSR_maskseq                     0.449 ± 0.000  0.432 ± 0.000   
ESM-MSR_seqonly                     0.502 ± 0.009  0.466 ± 0.010   
ESM3-large                               0.303242       0.353427   
ESM3-medium                              0.435070       0.405700   
ESM3-small                               0.441203       0.422059   
ESM3-small-open                          0.449304       0.432369   
MutateEverything                    0.425 ± 0.012  0.415 ± 0.001   
ProteinMPNN                              0.332573       0.286193   
Rosetta Cartesian DDG               0.202 ± 0.006  0.173 ± 0.003   
ThermoMPNN(-D)                      0.174 ± 0.030  0.249 ± 0.015   

                          Domainome1                 
                           ungrouped        grouped  
ESM-MSR                0.523 ± 0.005  0.537 ± 0.002  
ESM-MSR_maskseq        0.530 ± 0.000  0.554 ± 0.000  
ESM-MSR_seqonly        0.529 ± 0.003  0.536 ± 0.003  
ESM3-large                       NaN            NaN  
ESM3-medium                      NaN            NaN  
ESM3-small                       NaN            NaN  
ESM3-small-open                  NaN            NaN  
MutateEverything                 NaN            NaN  
ProteinMPNN                      NaN            NaN  
Rosetta Cartesian DDG            NaN            NaN  
ThermoMPNN(-D)                   NaN            NaN

In [21]:
# ==========================================
# 1. Data Processing Function
# ==========================================
def process_data(df):
    """
    Groups replicates by index suffix (e.g., _1, _2), 
    calculates Mean and Std, and returns a formatted string DataFrame.
    """
    df = df.copy()
    
    # 1. Handle Index: Remove suffixes like _1, _2
    if df.index.name is None:
        # Assume first column is index if not set
        df.index.name = "ID" 
    
    # regex to strip _\d+ from the end
    base_names = df.index.astype(str).str.replace(r'_\d+$', '', regex=True)

    # 2. Group and Aggregate
    grouped = df.groupby(base_names)
    means = grouped.mean()
    stds = grouped.std()

    # 3. Format as "Mean $\pm$ Std"
    # We keep the structure of the means dataframe
    formatted_df = pd.DataFrame(index=means.index, columns=means.columns)
    
    for col in means.columns:
        m_col = means[col]
        s_col = stds[col]
        
        formatted_list = []
        for m, s in zip(m_col, s_col):
            if pd.isna(s): 
                # Single replicate case
                formatted_list.append(f"{m:.3f}")
            else:
                # Mean +/- Std
                formatted_list.append(f"{m:.3f} $\pm$ {s:.3f}")
                
        formatted_df[col] = formatted_list
        
    return formatted_df

# ==========================================
# 2. Custom LaTeX Generator (The Fix)
# ==========================================
def generate_latex_manual(df):
    """
    Manually constructs a LaTeX table to avoid Pandas to_latex pitfalls.
    Handles MultiIndex headers and special characters correctly.
    """
    
    # Helper to escape LaTeX special characters (underscore, %, etc.)
    def esc(text):
        if not isinstance(text, str): return str(text)
        return text.replace('_', r'\_').replace('%', r'\%').replace('#', r'\#')

    lines = []
    
    # --- A. Setup Table ---
    # Determine number of columns (Index + Data columns)
    n_cols = len(df.columns) + 1 
    col_format = "l" + "c" * (n_cols - 1)
    
    lines.append(r"\begin{table}[h]")
    lines.append(r"\centering")
    lines.append(r"\caption{Aggregated Results (Mean $\pm$ Std)}")
    lines.append(r"\label{tab:results}")
    lines.append(r"\resizebox{\textwidth}{!}{%") # Optional: Resize to fit page
    lines.append(r"\begin{tabular}{" + col_format + "}")
    lines.append(r"\toprule")

    # --- B. Generate Headers ---
    # We assume a 2-level MultiIndex for columns based on your example
    if isinstance(df.columns, pd.MultiIndex):
        levels = df.columns.levels
        codes = df.columns.codes
        
        # 1. Top Header Row (Handling Spans)
        top_row = []
        # Add empty cell for the Index column header
        top_row.append(r"{}") 
        
        # Iterate through the top level codes to find spans
        current_code = codes[0][0]
        span_count = 0
        
        for code in codes[0]:
            if code == current_code:
                span_count += 1
            else:
                # Append the previous group
                label = esc(levels[0][current_code])
                top_row.append(r"\multicolumn{" + str(span_count) + r"}{c}{" + label + r"}")
                # Reset
                current_code = code
                span_count = 1
        # Append final group
        label = esc(levels[0][current_code])
        top_row.append(r"\multicolumn{" + str(span_count) + r"}{c}{" + label + r"}")
        
        lines.append(" & ".join(top_row) + r" \\")
        
        # 2. CMidrules (Optional cosmetic lines)
        # Calculate positions for cmidrules matches the spans above
        cmid_cmds = []
        current_col_idx = 2 # Start at column 2 (1-based, skipping index)
        current_code = codes[0][0]
        span_count = 0
        
        for code in codes[0]:
            if code == current_code:
                span_count += 1
            else:
                cmid_cmds.append(r"\cmidrule(lr){" + f"{current_col_idx}-{current_col_idx+span_count-1}" + "}")
                current_col_idx += span_count
                current_code = code
                span_count = 1
        # Final cmidrule
        cmid_cmds.append(r"\cmidrule(lr){" + f"{current_col_idx}-{current_col_idx+span_count-1}" + "}")
        lines.append(" ".join(cmid_cmds))

        # 3. Second Header Row (Sub-groups)
        second_row = [esc(df.index.name) if df.index.name else "Group"]
        for code in codes[1]:
            second_row.append(esc(levels[1][code]))
        lines.append(" & ".join(second_row) + r" \\")

    else:
        # Simple flat header fallback
        headers = [esc(c) for c in df.columns]
        lines.append(" & ".join([r"{}"] + headers) + r" \\")

    lines.append(r"\midrule")

    # --- C. Generate Data Rows ---
    for idx, row in df.iterrows():
        # Escape the index name
        row_str = [esc(idx)]
        # Data values already contain math syntax ($\pm$), do NOT escape them.
        row_str.extend(row.astype(str).tolist())
        lines.append(" & ".join(row_str) + r" \\")

    # --- D. Close Table ---
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}") # End resizebox
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

In [22]:
final_df = process_data(df3.join(df2_domainome))
index = ['ESM-MSR', 'ESM-MSR_paper', 'ESM-MSR_mut_ctx', 'ESM-MSR_nomutstructs', 'ESM-MSR_parallel', 'ESM-MSR_unmask_pos', 'ESM-MSR_mm', 'ESM-MSR_direct', 'ESM-MSR_huber', 'ESM-MSR_qkv_only', 'ESM-MSR_qkv_only', 'ESM-MSR_nobias', 'ESM-MSR_pool', 'ESM-MSR_singles_only', 'ESM-MSR_lessmutctx'] #,  'ESM-MSR_ll'
final_df = final_df.loc[index]
final_df = final_df.iloc[:, 2:]
final_df.rename({'ESM-MSR': 'ESM-MSR (masked chain)',
                    'ESM-MSR_mut_ctx': 'ESM-MSR, mut ctx structs',
                    'ESM-MSR_nomutstructs': 'Retrain, no mut ctx structs',
                    'ESM-MSR_parallel': 'ESM-MSR, no masking',
                    'ESM-MSR_unmask_pos': 'Retrain, no masking', 
                    'ESM-MSR_mm': 'ESM-MSR, masked marginal', 
                    'ESM-MSR_direct': 'Retrain, masked marginal', 
                    'ESM-MSR_huber': 'Retrain, Huber only',
                    'ESM-MSR_qkv': 'Retrain, add QKV',  
                    'ESM-MSR_qkv_only': 'Retrain, QKV only', 
                    'ESM-MSR_nobias': 'Retrain, no cal bias', 
                    'ESM-MSR_pool': 'Retrain, pool domains', 
                    'ESM-MSR_singles_only': 'Retrain, only singles', 
                    'ESM-MSR_ll': 'Retrain, log-likelihoods'}, inplace=True)
final_df

KeyError: "['ESM-MSR_paper', 'ESM-MSR_mut_ctx', 'ESM-MSR_nomutstructs', 'ESM-MSR_parallel', 'ESM-MSR_unmask_pos', 'ESM-MSR_mm', 'ESM-MSR_direct', 'ESM-MSR_huber', 'ESM-MSR_qkv_only', 'ESM-MSR_nobias', 'ESM-MSR_pool', 'ESM-MSR_singles_only', 'ESM-MSR_lessmutctx'] not in index"

In [ ]:
latex_code = generate_latex_manual(final_df)
print(latex_code)

\begin{table}[h]
\centering
\caption{Aggregated Results (Mean $\pm$ Std)}
\label{tab:results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccccc}
\toprule
{} & \multicolumn{2}{c}{feb19\_tsuboyama-S-test} & \multicolumn{2}{c}{feb19\_tsuboyama-D-test} & \multicolumn{2}{c}{feb19\_tsuboyama-D-test-dddG} & \multicolumn{2}{c}{Domainome1} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
ID & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped \\
\midrule
ESM-MSR (masked chain) & 0.765 $\pm$ 0.007 & 0.820 $\pm$ 0.004 & 0.586 $\pm$ 0.004 & 0.678 $\pm$ 0.005 & 0.546 $\pm$ 0.004 & 0.521 $\pm$ 0.006 & 0.523 $\pm$ 0.005 & 0.537 $\pm$ 0.002 \\
ESM-MSR, mut ctx structs & 0.765 $\pm$ 0.007 & 0.820 $\pm$ 0.004 & 0.586 $\pm$ 0.005 & 0.679 $\pm$ 0.006 & 0.551 $\pm$ 0.007 & 0.520 $\pm$ 0.007 & nan & nan \\
Retrain, no mut ctx structs & 0.759 $\pm$ 0.008 & 0.815 $\pm$ 0.004 & 0.599 $\pm$ 0.004 & 0.675 $\pm$ 0.005 & 0.547 $\pm$ 0.001 & 0.524